### Init model from config.yaml (and checkpoint)

In [ ]:
import os
import yaml

import hydra
import torch
import transformers
from transformers import AutoModel
from composer.models import write_huggingface_pretrained_from_composer_checkpoint
from omegaconf import OmegaConf
from src.utils import save_pretrained_or_push_to_hub

from scripts.utils import load_model_from_ckpt_dir_path

In [9]:
path_to_ckpt_dir = "/share/kuleshov/ma2238/runs/dllm-dev/gsm8k-0shot_block1024_lr1e-5_bsz1_warm100ba_alphaf0.5_max-dur75000ba_amp_bf16_layers28_aoarm_tgt4_max1024_distill_again_v2"  # TODO: Fill in ckpt_dir path
with open(os.path.join(path_to_ckpt_dir, "config.yaml"), "rb") as f:
    config = yaml.safe_load(f)
config = OmegaConf.create(config)

In [10]:
tokenizer = hydra.utils.instantiate(config.tokenizer)

In [11]:
ckpt_file = "best-rank0.pt"
use_ema = True
model = load_model_from_ckpt_dir_path(
    path_to_ckpt_dir=path_to_ckpt_dir,
    ckpt_file=ckpt_file,
    load_ema_weights=use_ema,
)

### Test push to hub

In [12]:
save_pretrained_or_push_to_hub(
    model=model,
    tokenizer=tokenizer,
    repo_id="kuleshov-group/setdlm-gsm8k-smax8",  # TODO: Set this to local path and `local=True` below to test local saving
    private=True,
    # local=True,  # TODO: Uncomment for testing local saving
)

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/1.91G [00:00<?, ?B/s]

#### Test round trip

In [ ]:
from transformers import AutoModelForMaskedLM

In [ ]:
test_model = AutoModelForMaskedLM.from_pretrained(
    "kuleshov-group/test-gsm8k-s32",  # TODO: Change to local path from above to test loading from locally saved checkpoint
    trust_remote_code=True,
)

In [ ]:
test_model